In [ ]:
from matplotlib import font_manager
import matplotlib.pyplot as plt

font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
font_manager.fontManager.addfont(font_path)
font_name = font_manager.FontProperties(fname=font_path).get_name()
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False
print("使用字体:", font_name)


# 📚 第三周-Day2：SFT监督微调的关键细节

> **预训练模型像一个博览群书但不会聊天的书呆子——你问它"今天天气如何"，它可能续写出"今天天气如何，这是一个常见的问题……"而不是直接回答。SFT（监督微调）就是教这个书呆子学会"有人问，你就答"的社交技能。**


## 📅 学习进度

| 阶段 | 状态 |
|------|------|
| W1：Transformer 基础架构 | ✅ 已完成 |
| W2：Transformer 深入理解 | ✅ 已完成 |
| **W3：大模型训练全景** | 🔄 **进行中（Day 2/7）** |
| W4-W12 | ⏳ 待开始 |

---


## 一、为什么需要 SFT？

### 1.1 预训练模型的"社交障碍"

预训练完成后，模型掌握了两项能力：
- ✅ 语言流畅度
- ✅ 广泛的世界知识

但它有两个致命问题：
- ❌ **不会遵循指令**：你说"请帮我写一封邮件"，它可能接着写"请帮我写一封辞职信……"
- ❌ **不会停止说话**：它习惯于无限续写，不会在回答结束后停下来

### 1.2 生活类比


In [ ]:
📖 预训练 = 读了万卷书的书呆子
    ↓ 知识丰富，但不会"做人"
🎓 SFT = 参加了社交礼仪培训
    教会他：
    • 别人问问题 → 要回答，不要续写
    • 遵循格式要求 → 要听话
    • 知道什么时候该闭嘴 → 学会停止
    ↓ 
🤝 结果 = 有知识 + 会沟通 = 可用的助手


### 1.3 SFT 的核心定义

**Supervised Fine-Tuning (SFT)**：使用高质量的"指令-回答"对，通过监督学习的方式调整预训练模型的参数，使其学会理解和遵循人类指令。

关键要素：
- **输入**：高质量的指令-回答对（Instruction-Response Pairs）
- **方法**：标准的监督学习（交叉熵损失）
- **输出**：能对话、能遵循指令的聊天模型
- **数据量**：通常 1万~10万 条（远少于预训练的万亿 token）

---


## 二、核心原理详解

### 2.1 SFT 的数据格式

SFT 最核心的变化是数据格式。从预训练的"纯文本"变成了"结构化对话"：

```json
{
  "instruction": "请解释什么是机器学习",
  "input": "",
  "output": "机器学习是人工智能的一个分支，它让计算机通过数据自动学习规律，而无需显式编程。"
}


In [ ]:
更复杂的格式还包括系统提示（System Prompt）：


json
{
  "messages": [
    {"role": "system", "content": "你是一个专业的客服助手。"},
    {"role": "user", "content": "你们的营业时间是几点？"},
    {"role": "assistant", "content": "我们的营业时间是每天10:00-22:00。"}
  ]
}


In [ ]:
### 2.2 数据质量 > 数据数量（LIMA 启示）

2023年Meta发表的LIMA论文震惊了业界：

> 仅用 **1000条** 精心人工撰写的高质量指令数据，就能把LLaMA微调到接近GPT-4的对话水平！

这告诉我们：

| 因素 | 影响程度 | 说明 |
|------|---------|------|
| 数据质量 | ⭐⭐⭐⭐⭐ | 决定了模型能力的上限 |
| 数据多样性 | ⭐⭐⭐⭐ | 覆盖各种任务和表达方式 |
| 数据数量 | ⭐⭐⭐ | 超过一定量后边际递减 |

**低质量数据的危害：**
- 含有错误答案 → 模型学会"胡说八道"
- 格式不统一 → 模型输出不稳定
- 指令过于单一 → 模型只会回答一类问题

### 2.3 Instruction Template：指令模板设计

指令模板是 SFT 的灵魂。好的模板能让模型明确任务边界：

**❌ 糟糕的模板：**


问：糖水店有什么
答：红豆沙绿豆沙


In [ ]:
**✅ 优秀的模板：**


你是一位专业的糖水店客服。请根据以下问题提供准确、友好的回答。

客户问题：{question}

回答要求：
1. 语气友好，使用敬语
2. 如果涉及价格，请给出具体数字
3. 如果不确定，请诚实说明

回答：


In [ ]:
**优秀模板的5要素：**
1. **角色设定**：明确模型扮演什么角色
2. **任务描述**：清晰说明要做什么
3. **背景信息**：提供必要的上下文
4. **输出格式**：指定回答的结构
5. **约束条件**：说明什么不能做

### 2.4 LoRA：参数高效微调

**LoRA (Low-Rank Adaptation)** 是目前SFT的主流方法。

**核心思想**：不修改原始模型的全部参数，而是在旁边加一个"小补丁"。

**生活类比**：


传统微调 = 把整本教材重新排版印刷（贵+慢）
LoRA    = 在教材旁边贴便利贴（便宜+快+不破坏原文）


In [ ]:
**数学原理（简化版）：**

原始权重矩阵 W (d×d)，LoRA 将更新量分解为两个小矩阵：


W_new = W_original + B @ A

其中：
  W_original: d×d  (冻结，不训练)
  A: r×d           (可训练，r通常=8或16)
  B: d×r           (可训练)
  r = 秩(rank)，远远小于 d


In [ ]:
**参数量对比**（以7B模型为例）：

| 方法 | 可训练参数 | 显存需求 | 训练速度 |
|------|----------|---------|---------|
| 全参数微调 | 7B (100%) | ~80GB | 慢 |
| LoRA (r=8) | ~18M (0.26%) | ~10GB | 快3-5倍 |
| QLoRA (4bit+LoRA) | ~18M (0.26%) | ~6GB | 快 |

### 2.5 SFT 训练过程详解

SFT 训练的标准流程：


1. 加载预训练模型
2. 准备指令数据集（格式化）
3. 配置训练参数（学习率、批次大小等）
4. 选择微调策略（全参数/LoRA/QLoRA）
5. 开始训练（通常3-5个epoch）
6. 在验证集上评估
7. 选择最佳checkpoint


In [ ]:
**SFT 特有的训练技巧：**
- **只对回答部分计算 Loss**：指令部分不参与Loss计算，只训练"回答"部分
- **短序列优先排序**：把长度相近的样本放一起，减少padding浪费
- **学习率较小**：通常 1e-5 ~ 5e-5（比预训练小10倍）

---

## 三、代码实战

### 3.1 创建指令数据集


python
import json

# 糖水店客服SFT数据示例
sft_data = [
    {
        "instruction": "糖水店的招牌产品是什么？",
        "input": "",
        "output": "我们的招牌产品是手工现熬的红豆沙和绿豆沙。选用优质红豆和绿豆，经过4小时慢火熬制，口感细腻，甜度适中，深受顾客喜爱！"
    },
    {
        "instruction": "你们的营业时间是什么？",
        "input": "",
        "output": "我们的营业时间是每天上午10:00到晚上22:00，全年无休。节假日不休息，欢迎您随时来品尝美味的糖水！"
    },
    {
        "instruction": "红豆沙多少钱一碗？",
        "input": "",
        "output": "红豆沙小碗8元，大碗12元。我们还有套餐优惠：红豆沙+杨枝甘露套餐只需25元，非常划算！"
    },
    {
        "instruction": "你们有外卖服务吗？",
        "input": "",
        "output": "有的！我们支持美团、饿了么外卖平台，配送范围3公里内。满30元免配送费，下单后30分钟内送达。"
    },
    {
        "instruction": "你们的产品含有什么过敏原？",
        "input": "",
        "output": "我们的部分产品含有牛奶（如杨枝甘露、芒果西米露）和花生（如花生糊）。如果您有过敏史，请提前告知店员，我们会为您推荐合适的产品。"
    }
]

print(f"📊 数据集大小：{len(sft_data)} 条")
print(f"📝 示例数据：\n{json.dumps(sft_data[0], ensure_ascii=False, indent=2)}")


In [ ]:
### 3.2 数据质量分析


python
import numpy as np
import matplotlib.pyplot as plt

# 设置中文字体
from matplotlib import font_manager
font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
font_manager.fontManager.addfont(font_path)
font_name = font_manager.FontProperties(fname=font_path).get_name()
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False

# 模拟数据质量与效果的关系
qualities = [0.1, 0.3, 0.5, 0.7, 0.9]  # 数据质量分数
quantities = [100, 500, 1000, 5000, 10000]  # 数据量

results = []
for q in qualities:
    for n in quantities:
        # 模型效果 ≈ 质量 × log(数量)
        effectiveness = q * np.log10(n + 1) * 20
        results.append({'quality': q, 'quantity': n, 'effect': effectiveness})

# 可视化
fig, ax = plt.subplots(figsize=(10, 6))
for q in qualities:
    qty_list = [r['quantity'] for r in results if r['quality'] == q]
    eff_list = [r['effect'] for r in results if r['quality'] == q]
    ax.plot(qty_list, eff_list, '-o', linewidth=2, markersize=6,
            label=f'质量分数={q}')

ax.set_xlabel('数据量（条）', fontsize=13)
ax.set_ylabel('模型效果分数', fontsize=13)
ax.set_title('SFT数据质量 vs 数量：质量决定上限！', fontsize=15)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_xscale('log')
plt.tight_layout()
plt.show()
print("💡 结论：高质量 + 少量 > 低质量 + 大量！")


In [ ]:
### 3.3 SFT 前后能力对比


python
# SFT前后能力雷达图
categories = ['指令理解', '对话连贯性', '知识应用', '安全性', '创造力', '格式遵循']
pre_sft = [20, 15, 70, 10, 35, 12]   # 预训练模型
post_sft = [88, 82, 72, 80, 68, 90]  # SFT后

angles = np.linspace(0, 2 * np.pi, len(categories), endpoint=False).tolist()
pre_sft += pre_sft[:1]
post_sft += post_sft[:1]
angles += angles[:1]

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
ax.plot(angles, pre_sft, 'r-o', linewidth=2, label='预训练模型')
ax.fill(angles, pre_sft, alpha=0.15, color='red')
ax.plot(angles, post_sft, 'b-s', linewidth=2, label='SFT后')
ax.fill(angles, post_sft, alpha=0.15, color='blue')

ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories, fontsize=12)
ax.set_title('SFT前后能力对比', fontsize=15, pad=20)
ax.legend(loc='upper right', fontsize=11)
plt.tight_layout()
plt.show()
print("✅ 指令理解和格式遵循提升最显著！")


In [ ]:
---

## 四、可视化理解

### 4.1 过拟合检测图


python
# SFT训练中的过拟合现象
epochs = list(range(1, 21))
train_loss = [2.8 * np.exp(-e/4) + 0.3 + 0.001*e**2 for e in epochs]
val_loss_normal = [2.6 * np.exp(-e/4) + 0.4 + 0.002*e**1.5 for e in epochs]
val_loss_overfit = [2.6 * np.exp(-e/4) + 0.4 + 0.003*e**1.8 for e in epochs]

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(epochs, train_loss, 'b-o', linewidth=2, label='训练集 Loss')
ax.plot(epochs, val_loss_normal, 'g-s', linewidth=2, label='验证集 Loss（正常）')
ax.plot(epochs, val_loss_overfit, 'r-^', linewidth=2, label='验证集 Loss（过拟合）')
ax.axvline(x=8, color='orange', linestyle='--', alpha=0.7, label='最佳停止点（epoch=8）')
ax.set_xlabel('Epoch', fontsize=13)
ax.set_ylabel('Loss', fontsize=13)
ax.set_title('SFT 过拟合检测：训练集一直降，验证集开始升就该停了！', fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
### 4.2 LoRA 参数量对比


python
# 不同微调方法的参数量对比
methods = ['全参数\n微调', 'LoRA\n(r=16)', 'LoRA\n(r=8)', 'QLoRA\n(4bit+r=8)']
trainable_params = [7000, 42, 21, 21]  # 单位：M (百万)
total_params = [7000, 7000, 7000, 7000]
frozen_params = [t - tr for t, tr in zip(total_params, trainable_params)]

fig, ax = plt.subplots(figsize=(10, 6))
x = range(len(methods))
width = 0.35

bars1 = ax.bar(x, frozen_params, width, label='冻结参数', color='lightblue')
bars2 = ax.bar(x, trainable_params, width, bottom=frozen_params, label='可训练参数', color='coral')

ax.set_ylabel('参数量 (M)', fontsize=13)
ax.set_title('7B模型不同微调方法的参数量对比', fontsize=15)
ax.set_xticks(x)
ax.set_xticklabels(methods, fontsize=11)
ax.legend(fontsize=12)
ax.set_yscale('log')

# 标注百分比
for i, (fz, tr) in enumerate(zip(frozen_params, trainable_params)):
    pct = tr / (fz + tr) * 100
    ax.text(i, fz + tr + 100, f'{pct:.1f}%', ha='center', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()
print("✅ LoRA 只训练 0.3% 的参数，就能达到接近全参数微调的效果！")


In [ ]:
---

## 五、业务关联

### 5.1 SFT 在糖水店AI助手中的落地路径


准备阶段：
  📊 收集历史客服对话 → 清洗标注 → 生成指令数据集

训练阶段：
  🔧 Qwen2-0.5B + LoRA + 4bit量化 → 消费级显卡就能跑

评估阶段：
  📈 用客服测试题验证 → 对比微调前后的回答质量

部署阶段：
  🚀 导出合并模型 → 部署到 LangChat → 7×24小时自动回复


In [ ]:
### 5.2 不同行业的 SFT 数据需求

| 行业 | 数据来源 | 数据量建议 | 特别注意 |
|------|---------|----------|---------|
| 餐饮客服 | 历史对话记录 | 500-2000条 | 注意方言和口语表达 |
| 电商售后 | 工单系统 | 2000-5000条 | 涵盖退换货全流程 |
| 金融咨询 | 合规问答库 | 5000+条 | 严格合规审查 |
| 医疗助手 | 医疗问答数据集 | 10000+条 | 必须有医生审核 |

### 5.3 和 LangChat/Agent 的关系

- **LangChat**：SFT 微调后的模型直接嵌入对话系统，提供更精准的领域回答
- **Agent**：SFT 训练的指令遵循能力是 Agent 执行多步任务的基础
- **企业AI**：SFT 是性价比最高的"定制化"手段——不需要从头训练，只需要几千条数据

---

## 六、常见误区

### ❌ 误区1："SFT数据越多越好"
**事实**：质量远比数量重要。LIMA用1000条数据就超过了别人用百万条的效果。关键是数据的多样性和准确性。

### ❌ 误区2："SFT后模型知识会增加"
**事实**：SFT 主要改变的是模型的"行为方式"（从续写到回答），而不是增加新知识。知识主要来自预训练。这就像教一个人"如何社交"，而不是教他"新的专业课"。

### ❌ 误区3："LoRA效果一定不如全参数微调"
**事实**：在大多数场景下，LoRA的效果接近全参数微调（差距<5%），但成本低10倍以上。除非你的任务和预训练领域差异巨大，否则LoRA是首选。

### ❌ 误区4："SFT数据不需要清洗"
**事实**：一条含有有害内容的指令数据就可能让模型"学坏"。数据清洗包括：去重、去除有害内容、格式统一、长度过滤。

### ❌ 误区5："学习率和预训练一样就行"
**事实**：SFT 的学习率要比预训练小一个数量级（1e-5 vs 1e-4），否则会"灾难性遗忘"预训练知识。

---

## 🧪 课堂练习（5分钟）

**练习1**：判断以下指令数据是高质量还是低质量，并说明理由：


A. {"instruction": "你好", "output": "你好！有什么可以帮助您的吗？"}
B. {"instruction": "写一首关于春天的诗", "output": "春眠不觉晓，处处闻啼鸟。"}
C. {"instruction": "计算 123+456", "output": "579"}
```

**练习2**：如果你要给糖水店做一个客服SFT数据集，你会包含哪些类型的问题？至少列出5类。

**练习3**：以下哪种情况可能是过拟合？
- A. 训练Loss持续下降，验证Loss也持续下降
- B. 训练Loss持续下降，验证Loss先降后升
- C. 训练Loss不变，验证Loss不变

---


## 📝 课后测试（15分钟）

**❶** SFT 的主要目标是什么？
- A. 增加模型的知识量
- B. 让模型学会遵循指令进行对话
- C. 减少模型参数量
- D. 提高推理速度

**❷** LoRA 的核心思想是？
- A. 完全替换原始模型权重
- B. 在原始权重旁加低秩矩阵补丁
- C. 剪枝掉不重要的参数
- D. 将模型量化到4bit

**❸** LIMA 论文证明了什么？
- A. 需要百万条数据才能做好SFT
- B. 1000条高质量数据就能达到很好效果
- C. 数据质量不重要
- D. LoRA比全参数微调好

**❹** SFT推荐的学习率范围是？
- A. 0.1 ~ 1.0
- B. 1e-5 ~ 5e-5
- C. 1e-10 ~ 1e-8
- D. 10 ~ 100

**❺** 简答题：为什么SFT通常只对"回答"部分计算Loss，而不对"指令"部分计算？

---


## 🔑 今日术语

| 英文 | 音标 | 中文 |
|------|------|------|
| Supervised Fine-Tuning (SFT) | /ˈsjuːpəvaɪzd faɪnˌtjuːnɪŋ/ | 监督微调 |
| Instruction Tuning | /ɪnˈstrʌkʃən ˈtjuːnɪŋ/ | 指令微调 |
| LoRA | /ˈloʊrə/ | 低秩适配（参数高效微调） |
| QLoRA | /kjuːˈloʊrə/ | 量化低秩适配 |
| Instruction Template | /ɪnˈstrʌkʃən ˈtɛmpleɪt/ | 指令模板 |
| Catastrophic Forgetting | /ˌkætəˈstrɒfɪk fərˈɡetɪŋ/ | 灾难性遗忘 |
| Epoch | /ˈɛpək/ | 训练轮次 |
| Overfitting | /ˌoʊvərˈfɪtɪŋ/ | 过拟合 |

---


## 📎 参考资源

### 必读论文
1. 📄 **LoRA: Low-Rank Adaptation of Large Language Models** (Hu et al., 2021)
   - https://arxiv.org/abs/2106.09685
2. 📄 **LIMA: Less Is More for Alignment** (Zhou et al., 2023)
   - https://arxiv.org/abs/2305.11206
3. 📄 **QLoRA: Efficient Finetuning of Quantized LLMs** (Dettmers et al., 2023)
   - https://arxiv.org/abs/2305.14314

### 视频推荐
1. 📺 **LoRA原理动画详解**（B站，约12分钟）
2. 📺 **HuggingFace PEFT库实战教程**（B站，约30分钟）

### 明日预告
明天进入 **RLHF（人类反馈强化学习）**！SFT让模型"会回答"，RLHF让模型"答得好"——它是ChatGPT成功的关键秘密武器 🚀
